# Modelling: ACS PUMS Income (2024)

Mirrors `04_modelling_uci.ipynb`: same `fit_and_evaluate`, same two models, same experiments and long-format export. Only the data and dataset-specific column choices differ, so any difference in results between this notebook and 04 is attributable to the datasets (1994 vs 2024), not the method.

**Differences from the UCI notebook, traceable to the ACS EDA (notebook 02):**
- Fewer numeric features: only `age` and `hours-per-week`.
- The proxies removed are `occupation` (top sex proxy, V = 0.451) and `birthplace-us` (top race proxy, V = 0.516). `relationship` collapsed as a sex proxy in 2024.
- Raw carry-along columns are dropped; `PINCP` (raw dollar income) in particular would be target leakage.

Results exported to `results/tables/acs_model_results.csv` for notebook 06.

## 0. Setup

Imports and the reusable `fit_and_evaluate` function : identical to notebook 04.

In [25]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

from fairlearn.metrics import (
    MetricFrame,
    selection_rate,
    true_positive_rate,
    false_positive_rate,
    demographic_parity_difference,
    equalized_odds_difference,
)

SEED = 42
DATASET = "ACS PUMS (2024)"
RESULTS_DIR = "../results/tables"

pd.set_option("display.width", 120)

In [26]:
def fit_and_evaluate(model, preprocessor, X_tr, y_tr, X_te, y_te,
                     sensitive_cols, model_name, experiment, dataset_name):
    """Fit a model, predict on the test set, and compute accuracy plus the five fairness
    metrics per protected attribute. Returns long-format rows for notebook 06.

    Copied verbatim from notebook 04 so both datasets run the same pipeline. Sensitive
    features are read from the raw X_te (pre-encoding).
    """
    X_tr_p = preprocessor.fit_transform(X_tr)
    X_te_p = preprocessor.transform(X_te)
    model.fit(X_tr_p, y_tr)
    y_pred = model.predict(X_te_p)

    rows = []
    acc = accuracy_score(y_te, y_pred)

    for attr in sensitive_cols:
        sf = X_te[attr].to_numpy()

        spd = demographic_parity_difference(y_te, y_pred, sensitive_features=sf)
        eqo = equalized_odds_difference(y_te, y_pred, sensitive_features=sf)

        mf = MetricFrame(
            metrics={
                "selection_rate": selection_rate,
                "tpr": true_positive_rate,
                "precision": precision_score,
            },
            y_true=y_te, y_pred=y_pred, sensitive_features=sf,
        )
        by = mf.by_group
        di = by["selection_rate"].min() / by["selection_rate"].max()
        eod = by["tpr"].max() - by["tpr"].min()
        pp = by["precision"].max() - by["precision"].min()

        for metric_name, value in [
            ("accuracy", acc),
            ("SPD", spd),
            ("DI", di),
            ("EOD", eod),
            ("EqualisedOdds", eqo),
            ("PredictiveParity", pp),
        ]:
            rows.append({
                "dataset": dataset_name,
                "model": model_name,
                "experiment": experiment,
                "protected_attribute": attr,
                "metric": metric_name,
                "value": round(float(value), 4),
            })

    return rows

## 1. Load and split

The prepared ACS frame from notebook 02 is loaded. A keep-list defines exactly which columns
are used for modelling; everything else : raw ACS codes, the dollar income `PINCP` (which
would be target leakage), and alternate race encodings : is dropped by exclusion. This is
safer than a drop-list, because any unexpected column is excluded by default rather than
silently entering the model.

In [27]:
df = pd.read_csv("../data/processed/acs_2024_prepared.csv")

# Target: ">50K" is the positive class (1); same mapping as UCI.
y = (df["income"] == ">50K").astype(int)

# Keep-list: only these columns are used as features. Anything not named is dropped, so raw
# codes (PINCP, RAC1P, RELP, OCCP, SCHL, POBP), the adjusted-threshold target, raw race and
# race_cluster, and the target never reach the model. PINCP is excluded deliberately: as raw
# dollar income it would leak the target.
NUMERIC = ["age", "hours-per-week"]
CATEGORICAL = ["workclass", "education", "marital-status", "occupation",
               "relationship", "birthplace-us", "sex", "race_binary"]
KEEP = NUMERIC + CATEGORICAL

missing = [c for c in KEEP if c not in df.columns]
assert not missing, f"expected feature columns missing from frame: {missing}"

X = df[KEEP].copy()
PROTECTED = ["sex", "race_binary"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

print("train:", X_train.shape, "| test:", X_test.shape)
print("train positive rate:", round(y_train.mean(), 4),
      "| test positive rate:", round(y_test.mean(), 4))
print("feature columns:", X.columns.tolist())

train: (131373, 10) | test: (43791, 10)
train positive rate: 0.4908 | test positive rate: 0.4907
feature columns: ['age', 'hours-per-week', 'workclass', 'education', 'marital-status', 'occupation', 'relationship', 'birthplace-us', 'sex', 'race_binary']


## 2. Preprocessing

Same design as notebook 04: numerics standardised, categoricals one-hot encoded, one preprocessor factory so each experiment fits its own transformer. `birthplace-us` is boolean and one-hot encodes to two indicators.

In [28]:
assert set(NUMERIC + CATEGORICAL) == set(X_train.columns), "column role mismatch"


def make_preprocessor(numeric, categorical):
    """Fresh ColumnTransformer per experiment (unfitted), identical design to notebook 04."""
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
        ],
        remainder="drop",
    )


_pp = make_preprocessor(NUMERIC, CATEGORICAL)
_tr = _pp.fit_transform(X_train)
_te = _pp.transform(X_test)
print("processed train:", _tr.shape, "| processed test:", _te.shape,
      "| equal cols:", _tr.shape[1] == _te.shape[1])

processed train: (131373, 71) | processed test: (43791, 71) | equal cols: True


## 3. Baseline

Both models trained on the full feature set (protected attributes included), evaluated with `fit_and_evaluate`. The reference point for the later experiments.

In [29]:
results = []  # accumulates every experiment's rows; exported at the end

lr = LogisticRegression(max_iter=1000, random_state=SEED)
results += fit_and_evaluate(
    lr, make_preprocessor(NUMERIC, CATEGORICAL),
    X_train, y_train, X_test, y_test,
    sensitive_cols=PROTECTED,
    model_name="LogisticRegression", experiment="baseline", dataset_name=DATASET,
)

rf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
results += fit_and_evaluate(
    rf, make_preprocessor(NUMERIC, CATEGORICAL),
    X_train, y_train, X_test, y_test,
    sensitive_cols=PROTECTED,
    model_name="RandomForest", experiment="baseline", dataset_name=DATASET,
)

baseline_df = pd.DataFrame(results)
baseline_pivot = baseline_df.pivot_table(
    index="metric", columns=["model", "protected_attribute"], values="value")
print(baseline_pivot.round(4))

model               LogisticRegression         RandomForest        
protected_attribute        race_binary     sex  race_binary     sex
metric                                                             
DI                              0.7054  0.6871       0.7352  0.7063
EOD                             0.1113  0.1072       0.0898  0.0985
EqualisedOdds                   0.1113  0.1102       0.0898  0.0985
PredictiveParity                0.0487  0.0344       0.0616  0.0509
SPD                             0.1637  0.1847       0.1408  0.1659
accuracy                        0.7843  0.7843       0.7629  0.7629


## 4. Protected attributes removed

Two forms, matching notebook 04:
- **Naive:** drop only `sex` and `race_binary`.
- **Plus proxies:** also drop `occupation` (top ACS sex proxy) and `birthplace-us` (top ACS race proxy). These proxies come from the ACS EDA and differ from the UCI proxies, which is the point of running per-dataset EDA before modelling.

In both, fairness is still audited on `sex` and `race_binary` (read from the raw test frame), even though the model no longer trains on them.

In [30]:
# naive: drop only the protected attributes.
NUM_B1 = [c for c in NUMERIC if c not in ("sex", "race_binary")]
CAT_B1 = [c for c in CATEGORICAL if c not in ("sex", "race_binary")]

for name, model in [("LogisticRegression", LogisticRegression(max_iter=1000, random_state=SEED)),
                    ("RandomForest", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))]:
    results += fit_and_evaluate(
        model, make_preprocessor(NUM_B1, CAT_B1),
        X_train, y_train, X_test, y_test,
        sensitive_cols=PROTECTED,
        model_name=name, experiment="protected_removed_naive", dataset_name=DATASET,
    )

# plus proxies: also drop the top ACS proxies (occupation for sex, birthplace-us for race).
PROXIES = ["occupation", "birthplace-us"]
NUM_B2 = [c for c in NUMERIC if c not in (["sex", "race_binary"] + PROXIES)]
CAT_B2 = [c for c in CATEGORICAL if c not in (["sex", "race_binary"] + PROXIES)]

for name, model in [("LogisticRegression", LogisticRegression(max_iter=1000, random_state=SEED)),
                    ("RandomForest", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))]:
    results += fit_and_evaluate(
        model, make_preprocessor(NUM_B2, CAT_B2),
        X_train, y_train, X_test, y_test,
        sensitive_cols=PROTECTED,
        model_name=name, experiment="protected_removed_proxies", dataset_name=DATASET,
    )

comp = pd.DataFrame(results)
sex_view = comp[comp["protected_attribute"] == "sex"].pivot_table(
    index="metric", columns=["experiment", "model"], values="value")
print("SEX across experiments:")
print(sex_view.round(4))

SEX across experiments:
experiment                 baseline              protected_removed_naive              protected_removed_proxies  \
model            LogisticRegression RandomForest      LogisticRegression RandomForest        LogisticRegression   
metric                                                                                                            
DI                           0.6871       0.7063                  0.8450       0.8166                    0.9343   
EOD                          0.1072       0.0985                  0.0036       0.0258                    0.0406   
EqualisedOdds                0.1102       0.0985                  0.0115       0.0304                    0.0406   
PredictiveParity             0.0344       0.0509                  0.0904       0.0856                    0.1186   
SPD                          0.1847       0.1659                  0.0841       0.0974                    0.0344   
accuracy                     0.7843       0.7629        

## 5. Export

All rows assembled into one long-format table and written for notebook 06. A duplicate guard catches accidental re-runs without a kernel restart.

In [31]:
acs_results = pd.DataFrame(results)

key = ["model", "experiment", "protected_attribute", "metric"]
dupes = acs_results.duplicated(subset=key).sum()
print("duplicate rows:", dupes, "(should be 0)")
if dupes:
    print("WARNING: results list contains duplicates; restart kernel and Run All to fix.")

print("experiments present:", sorted(acs_results["experiment"].unique()))
print("total rows:", len(acs_results))

acs_results.to_csv(f"{RESULTS_DIR}/acs_model_results.csv", index=False)
print(f"written: {RESULTS_DIR}/acs_model_results.csv")

duplicate rows: 0 (should be 0)
experiments present: ['baseline', 'protected_removed_naive', 'protected_removed_proxies']
total rows: 72
written: ../results/tables/acs_model_results.csv


## 6. Additional analyses (threshold, bootstrap, intersectional, multi-seed)

Each subsection writes its own CSV and leaves the main `acs_model_results.csv` export untouched, so the baseline pipeline is unaffected.

### 6.1 Threshold sensitivity (ACS baseline)

In [32]:
# Threshold sensitivity (ACS baseline only).
# The $50k threshold means different things across 30 years: in 1994, >$50k was
# the top 23.93% of earners; in 2024 the same nominal $50k is ~49%, a different
# problem with a different base rate. The ACS baseline is re-run at three
# thresholds so the temporal comparison is defensible.
UCI_1994_POS_RATE = 0.2393   # 1994 train/test positive rate

df_thr = pd.read_csv("../data/processed/acs_2024_prepared.csv")   # needs PINCP (dropped from X)

thresholds = {
    "nominal_50k":       50_000,
    "inflation_105833":  105_833,
    "quantile_matched":  float(df_thr["PINCP"].quantile(1 - UCI_1994_POS_RATE)),
}
print("Threshold            dollar cutoff   ACS positive rate")
for nm, cut in thresholds.items():
    print(f"{nm:18} ${cut:>10,.0f}    {(df_thr['PINCP'] > cut).mean():.4f}")

# Re-run the baseline (LR + RF) at each threshold. Only the target changes; the
# feature matrix and the split are identical to the main baseline, so the rows
# are directly comparable. The same stratified 75/25 split is used each time.
thr_rows = []
for tname, cut in thresholds.items():
    y_thr = (df_thr["PINCP"] > cut).astype(int)
    Xtr_t, Xte_t, ytr_t, yte_t = train_test_split(
        X, y_thr, test_size=0.25, random_state=SEED, stratify=y_thr)
    for mname, model in [("LogisticRegression", LogisticRegression(max_iter=1000, random_state=SEED)),
                         ("RandomForest", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))]:
        thr_rows += fit_and_evaluate(
            model, make_preprocessor(NUMERIC, CATEGORICAL),
            Xtr_t, ytr_t, Xte_t, yte_t,
            sensitive_cols=PROTECTED, model_name=mname,
            experiment=f"threshold_{tname}", dataset_name=DATASET)

thr_df = pd.DataFrame(thr_rows)
print("\nThreshold sensitivity: SEX, Disparate Impact & SPD:")
view = thr_df[thr_df["protected_attribute"]=="sex"].pivot_table(
    index="metric", columns=["experiment","model"], values="value")
print(view.round(4))
thr_df.to_csv(f"{RESULTS_DIR}/acs_threshold_results.csv", index=False)
print(f"\nwritten: {RESULTS_DIR}/acs_threshold_results.csv")


Threshold            dollar cutoff   ACS positive rate
nominal_50k        $    50,000    0.4908
inflation_105833   $   105,833    0.1784
quantile_matched   $    90,000    0.2350

Threshold sensitivity: SEX, Disparate Impact & SPD:
experiment       threshold_inflation_105833              threshold_nominal_50k               \
model                    LogisticRegression RandomForest    LogisticRegression RandomForest   
metric                                                                                        
DI                                   0.2696       0.4410                0.6871       0.7063   
EOD                                  0.2826       0.1696                0.1072       0.0985   
EqualisedOdds                        0.2826       0.1696                0.1102       0.0985   
PredictiveParity                     0.0367       0.1050                0.0344       0.0509   
SPD                                  0.1316       0.1070                0.1847       0.1659   
accuracy 

### 6.2 Bootstrap 95% confidence intervals (baseline)

In [33]:
# Bootstrap 95% CIs for the fairness metrics.
# The test set is resampled with replacement, each metric is recomputed per
# resample, and the 2.5th and 97.5th percentiles are taken. The model is trained
# once; only the evaluation is resampled.
from sklearn.metrics import precision_score
from fairlearn.metrics import (MetricFrame, selection_rate,
                               true_positive_rate, demographic_parity_difference,
                               equalized_odds_difference)

def bootstrap_fairness_cis(model, preprocessor, X_tr, y_tr, X_te, y_te,
                           sensitive_cols, model_name, experiment, dataset_name,
                           n_boot=200, seed=42):
    """Train once, then bootstrap the test set n_boot times. Returns long-format rows:
    {dataset, model, experiment, protected_attribute, metric, value,
     ci_low, ci_high, n_test, n_pos}."""
    rng = np.random.default_rng(seed)

    X_tr_p = preprocessor.fit_transform(X_tr)
    X_te_p = preprocessor.transform(X_te)
    model.fit(X_tr_p, y_tr)

    y_te = np.asarray(y_te)
    idx_all = np.arange(len(y_te))

    def metrics_on(idx, y_pred_full):
        yt = y_te[idx]; yp = y_pred_full[idx]
        out = {}
        for attr in sensitive_cols:
            sf = X_te[attr].to_numpy()[idx]
            
            if len(np.unique(sf)) < 2:
                out[attr] = None; continue
            spd = demographic_parity_difference(yt, yp, sensitive_features=sf)
            eqo = equalized_odds_difference(yt, yp, sensitive_features=sf)
            mf = MetricFrame(metrics={"sel": selection_rate, "tpr": true_positive_rate,
                                      "prec": lambda yt,yp: precision_score(yt,yp,zero_division=0)},
                             y_true=yt, y_pred=yp, sensitive_features=sf)
            by = mf.by_group
            di  = by["sel"].min() / by["sel"].max() if by["sel"].max() > 0 else np.nan
            eod = by["tpr"].max() - by["tpr"].min()
            pp  = by["prec"].max() - by["prec"].min()
            out[attr] = {"SPD": spd, "DI": di, "EOD": eod,
                         "EqualisedOdds": eqo, "PredictiveParity": pp}
        return out

    y_pred_full = model.predict(X_te_p)
    point = metrics_on(idx_all, y_pred_full)

    # bootstrap
    boot = {attr: {m: [] for m in ["SPD","DI","EOD","EqualisedOdds","PredictiveParity"]}
            for attr in sensitive_cols}
    for _ in range(n_boot):
        idx = rng.choice(idx_all, size=len(idx_all), replace=True)
        res = metrics_on(idx, y_pred_full)
        for attr in sensitive_cols:
            if res[attr] is None: continue
            for m, v in res[attr].items():
                if v is not None and not np.isnan(v):
                    boot[attr][m].append(v)

    rows = []
    for attr in sensitive_cols:
        sf_full = X_te[attr].to_numpy()
        n_test = len(sf_full)
        n_pos = int(y_te.sum())
        for m in ["SPD","DI","EOD","EqualisedOdds","PredictiveParity"]:
            vals = np.array(boot[attr][m])
            lo, hi = (np.percentile(vals, [2.5, 97.5]) if len(vals) else (np.nan, np.nan))
            rows.append({"dataset": dataset_name, "model": model_name,
                         "experiment": experiment, "protected_attribute": attr,
                         "metric": m, "value": round(float(point[attr][m]), 4),
                         "ci_low": round(float(lo), 4), "ci_high": round(float(hi), 4),
                         "n_test": n_test, "n_pos": n_pos})
    return rows

print("bootstrap_fairness_cis defined")


bootstrap_fairness_cis defined


In [34]:
CI_FILE = "acs_ci_baseline.csv"
# Bootstrap CIs on the baseline models (the reference the results are read against).
# n_boot=200: the ACS test set is ~44k rows, so 1000 resamples took ~50 min for no
# meaningful gain in interval stability. 200 is ample for 95% CIs at this sample size.
ci_rows = []
for name, model in [("LogisticRegression", LogisticRegression(max_iter=1000, random_state=SEED)),
                    ("RandomForest", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))]:
    ci_rows += bootstrap_fairness_cis(
        model, make_preprocessor(NUMERIC, CATEGORICAL),
        X_train, y_train, X_test, y_test,
        sensitive_cols=PROTECTED, model_name=name,
        experiment="baseline", dataset_name=DATASET,
        n_boot=200, seed=SEED)

ci_df = pd.DataFrame(ci_rows)
print("Baseline fairness metrics with 95% bootstrap CIs:")
print(ci_df[["model","protected_attribute","metric","value","ci_low","ci_high","n_test","n_pos"]].to_string(index=False))
ci_df.to_csv(f"{RESULTS_DIR}/{CI_FILE}", index=False)
print(f"\nwritten: {RESULTS_DIR}/{CI_FILE}") 

Baseline fairness metrics with 95% bootstrap CIs:
             model protected_attribute           metric  value  ci_low  ci_high  n_test  n_pos
LogisticRegression                 sex              SPD 0.1847  0.1760   0.1923   43791  21490
LogisticRegression                 sex               DI 0.6871  0.6760   0.7006   43791  21490
LogisticRegression                 sex              EOD 0.1072  0.0981   0.1161   43791  21490
LogisticRegression                 sex    EqualisedOdds 0.1102  0.1048   0.1199   43791  21490
LogisticRegression                 sex PredictiveParity 0.0344  0.0230   0.0469   43791  21490
LogisticRegression         race_binary              SPD 0.1637  0.1539   0.1735   43791  21490
LogisticRegression         race_binary               DI 0.7054  0.6895   0.7206   43791  21490
LogisticRegression         race_binary              EOD 0.1113  0.0981   0.1229   43791  21490
LogisticRegression         race_binary    EqualisedOdds 0.1113  0.0981   0.1229   43791  21490


### 6.3 Intersectional race × sex (baseline)

In [35]:
INTER_FILE = "acs_intersectional.csv"
# Intersectional analysis: race x sex on the baseline models.
# A model that is fair on race and on sex separately can still be unfair to a
# specific combination (e.g. Black women). A combined race_binary x sex group is
# built and the baseline models are audited on it.
from fairlearn.metrics import MetricFrame, selection_rate, true_positive_rate
from sklearn.metrics import precision_score

def intersectional_eval(model, preprocessor, X_tr, y_tr, X_te, y_te,
                        model_name, dataset_name):
    """Audit one trained baseline model across race_binary × sex subgroups.
    Returns per-subgroup rows plus overall max-gap rows."""
    X_tr_p = preprocessor.fit_transform(X_tr)
    X_te_p = preprocessor.transform(X_te)
    model.fit(X_tr_p, y_tr)
    y_pred = model.predict(X_te_p)
    yt = np.asarray(y_te)

    inter = (X_te["race_binary"].astype(str) + " / " + X_te["sex"].astype(str)).to_numpy()
    mf = MetricFrame(metrics={"selection_rate": selection_rate,
                              "tpr": true_positive_rate,
                              "precision": lambda yt,yp: precision_score(yt,yp,zero_division=0)},
                     y_true=yt, y_pred=y_pred, sensitive_features=inter)
    by = mf.by_group

    rows = []
    for grp in by.index:
        n = int((inter == grp).sum())
        rows.append({"dataset": dataset_name, "model": model_name,
                     "experiment": "baseline_intersectional", "subgroup": grp,
                     "n": n,
                     "selection_rate": round(float(by.loc[grp, "selection_rate"]), 4),
                     "tpr": round(float(by.loc[grp, "tpr"]), 4),
                     "precision": round(float(by.loc[grp, "precision"]), 4)})
    # overall gaps across the four subgroups
    rows.append({"dataset": dataset_name, "model": model_name,
                 "experiment": "baseline_intersectional", "subgroup": "GAP (max-min)",
                 "n": len(inter),
                 "selection_rate": round(float(by["selection_rate"].max() - by["selection_rate"].min()), 4),
                 "tpr": round(float(by["tpr"].max() - by["tpr"].min()), 4),
                 "precision": round(float(by["precision"].max() - by["precision"].min()), 4)})
    return rows

inter_rows = []
for name, model in [("LogisticRegression", LogisticRegression(max_iter=1000, random_state=SEED)),
                    ("RandomForest", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))]:
    inter_rows += intersectional_eval(
        model, make_preprocessor(NUMERIC, CATEGORICAL),
        X_train, y_train, X_test, y_test, name, DATASET)

inter_df = pd.DataFrame(inter_rows)
print("Intersectional (race_binary x sex) baseline:")
print(inter_df.to_string(index=False))
inter_df.to_csv(f"{RESULTS_DIR}/{INTER_FILE}", index=False)
print(f"\nwritten: {RESULTS_DIR}/{INTER_FILE}")


Intersectional (race_binary x sex) baseline:
        dataset              model              experiment           subgroup     n  selection_rate    tpr  precision
ACS PUMS (2024) LogisticRegression baseline_intersectional Non-White / Female  7175          0.3242 0.6613     0.7386
ACS PUMS (2024) LogisticRegression baseline_intersectional   Non-White / Male  7487          0.4569 0.7479     0.7387
ACS PUMS (2024) LogisticRegression baseline_intersectional     White / Female 14036          0.4473 0.7552     0.7593
ACS PUMS (2024) LogisticRegression baseline_intersectional       White / Male 15093          0.6565 0.8671     0.8051
ACS PUMS (2024) LogisticRegression baseline_intersectional      GAP (max-min) 43791          0.3323 0.2058     0.0665
ACS PUMS (2024)       RandomForest baseline_intersectional Non-White / Female  7175          0.3398 0.6605     0.7039
ACS PUMS (2024)       RandomForest baseline_intersectional   Non-White / Male  7487          0.4396 0.7076     0.7265
ACS PUMS (2

### 6.4 Multi-seed subsample robustness

In [36]:
# Multi-seed ACS subsample robustness.
# Is a 1994 to 2024 difference real, or just sampling noise in the ACS subsample?
# The ACS sample is matched to the 1994 dataset size and drawn under several seeds
# using stratified sampling (preserving population representation, not equalising
# groups). The baseline is re-run each time and the spread of the fairness metrics
# across seeds is examined.
from sklearn.model_selection import train_test_split as _tts

UCI_1994_N = 48842          # full 1994 dataset size (train+test)
SEEDS = [0, 1, 2, 3, 4]     # 5 seeds

# Stratify the subsample on race_binary x sex x target so representation is kept.
strat_key = (X["race_binary"].astype(str) + "|" + X["sex"].astype(str) + "|" + y.astype(str))

ms_rows = []
for s in SEEDS:
    n = min(UCI_1994_N, len(X))
    # stratified draw of size n, preserving the joint distribution
    X_s, _, y_s, _, strat_s, _ = _tts(
        X, y, strat_key, train_size=n, random_state=s, stratify=strat_key)
    # inner split varies with the seed too, so each iteration is fully independent
    Xtr_s, Xte_s, ytr_s, yte_s = _tts(
        X_s, y_s, test_size=0.25, random_state=s, stratify=y_s)
    for mname, model in [("LogisticRegression", LogisticRegression(max_iter=1000, random_state=s)),
                         ("RandomForest", RandomForestClassifier(n_estimators=300, random_state=s, n_jobs=-1))]:
        rows = fit_and_evaluate(
            model, make_preprocessor(NUMERIC, CATEGORICAL),
            Xtr_s, ytr_s, Xte_s, yte_s,
            sensitive_cols=PROTECTED, model_name=mname,
            experiment="baseline_multiseed", dataset_name=DATASET)
        for r in rows:
            r["seed"] = s
        ms_rows += rows

ms_df = pd.DataFrame(ms_rows)
# summarise spread across seeds: mean and std per (model, attr, metric)
summary = (ms_df.groupby(["model","protected_attribute","metric"])["value"]
                 .agg(["mean","std","min","max"]).round(4).reset_index())
print("Multi-seed baseline (ACS matched to 1994 size, 5 seeds): spread per metric:")
print(summary.to_string(index=False))
ms_df.to_csv(f"{RESULTS_DIR}/acs_multiseed_results.csv", index=False)
summary.to_csv(f"{RESULTS_DIR}/acs_multiseed_summary.csv", index=False)
print(f"\nwritten: {RESULTS_DIR}/acs_multiseed_results.csv  and  _summary.csv")

Multi-seed baseline (ACS matched to 1994 size, 5 seeds): spread per metric:
             model protected_attribute           metric   mean    std    min    max
LogisticRegression         race_binary               DI 0.6966 0.0196 0.6810 0.7290
LogisticRegression         race_binary              EOD 0.1207 0.0255 0.0870 0.1491
LogisticRegression         race_binary    EqualisedOdds 0.1207 0.0255 0.0870 0.1491
LogisticRegression         race_binary PredictiveParity 0.0300 0.0064 0.0225 0.0391
LogisticRegression         race_binary              SPD 0.1704 0.0121 0.1503 0.1795
LogisticRegression         race_binary         accuracy 0.7867 0.0026 0.7843 0.7906
LogisticRegression                 sex               DI 0.6943 0.0112 0.6745 0.7008
LogisticRegression                 sex              EOD 0.0983 0.0136 0.0851 0.1164
LogisticRegression                 sex    EqualisedOdds 0.1108 0.0105 0.0965 0.1256
LogisticRegression                 sex PredictiveParity 0.0360 0.0146 0.0223 0.0567
